In [1]:
# ------------------- Import packages --------------------
import pandas as pd
import numpy as np
import glob, os, json
import polars as pl
from datetime import datetime
import locale
locale.setlocale(locale.LC_TIME, 'en_US.UTF-8')

'en_US.UTF-8'

In [2]:
# ----------------------- 01 -Load labels -------------------------
with open("Labels_Set.json", "r") as f: all_labels_set = set(json.load(f))

# ------------------------- 02 - Addressable Files ---------------------------
addressable_files = glob.glob(os.path.join('Data/Addressable_Data', "*.csv"))

# Sort files by date
def get_file_date(f):
    df = pl.read_csv(f, n_rows=1)
    raw_date = str(df["date_time"][0]).split()[0]
    dt = datetime.strptime(raw_date, "%Y-%m-%d")
    return dt

addressable_files = sorted(addressable_files, key=get_file_date)

# ---------------------- 03 - Load and prepare Addressable file ----------------------
def load_addressable_file(f):
    """Load CSV once, return filtered and unfiltered versions."""
    df = pl.read_csv(f).rename({"Total $ Balance in Eth": "ETH","Total $ balance in Wrapped Ether": "WETH","Total $ balance in stETH-like": "stETH"})
    df = df.with_columns([pl.col(["ETH","WETH","stETH"]).cast(pl.Float64).fill_null(0)])  # Convert columns to float and fill nulls
    df = df.with_columns([(pl.col("ETH") + pl.col("WETH") + pl.col("stETH")).alias("Total")])     # Compute total
    dt = datetime.strptime(str(df["date_time"][0]).split()[0], "%Y-%m-%d")  # Extract date
    df_total = df.sort("Total", descending=True).head(min(1_000_000, df.height))  # Top 1M wallets according to total
    df_eth = df.sort("ETH", descending=True).head(min(1_000_000, df.height))  # Top 1M wallets according to eth
    df_filtered_total = df_total.filter(~df_total["address"].str.strip_chars(" ").str.to_lowercase().is_in(all_labels_set)) # Filtered version, total
    df_filtered_eth = df_eth.filter(~df_eth["address"].str.strip_chars(" ").str.to_lowercase().is_in(all_labels_set)) # Filtered version, eth
    return df_filtered_total, df_total, df_filtered_eth, df_eth, dt


# ---------------------- 04 - Load and prepare Artemis file ------------------------
artemis_files = sorted(glob.glob(os.path.join('Data/Artemis_Data', "*.csv")))

def load_artemis_file(f):
    """Load Artemis CSV once, return filtered and unfiltered versions."""
    df = pl.read_csv(f)
    df = df.rename({"ADDRESS": "address", "BALANCE_TOKEN": "ETH"})
    df = df.drop(["CONTRACT_ADDRESS", "BLOCK_TIMESTAMP"])
    df = df.sort("ETH", descending=True).head(min(1_000_000, df.height)) # Top 1M wallets
    df_filtered = df.filter(~df["address"].str.strip_chars(" ").str.to_lowercase().is_in(all_labels_set))  # Filtered version 
    date_str = os.path.splitext(os.path.basename(f))[0][:10] # Extract date from filename
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return df_filtered, df, dt

# ----------- 05 - Function: Gini coefficient calculation ----------
def gini(w: list[float]) -> float:
    """Given a list of wealth values w, returns the Gini coefficient."""
    if len(w) == 0: return np.nan
    if np.sum(w) == 0: return np.nan
    return (2 * np.sum(np.arange(1, len(w) + 1) * np.sort(w)) / np.sum(w) - 1) / len(w) - 1

# --------------------- 06 - Function: Random Gini -------------------
sample_size = 10000
n_iterations = 1000

def gini_random(balances, n_iterations=n_iterations, sample_size=sample_size):
    """Returns the average Gini coefficient and confidence intervals from repeated random sampling."""
    np.random.seed(123)
    samples = np.random.choice(balances, size=(n_iterations, sample_size), replace=True)
    gini_samples = np.array([gini(row) for row in samples])
    return np.mean(gini_samples), np.percentile(gini_samples, 2.5), np.percentile(gini_samples, 97.5)

# ------------------ 07 - Define Rankings ------------------
rankings = [380, 1600]


In [3]:
# ----------------------------------- Compute Gini & Filtered Wallets -----------------------------------

# ---------------- 01 - Addressable Gini & Filtered Wallets -----------------
results_addressable = []

for f in addressable_files:
    df_filtered_total, df_unfiltered_total,df_filtered_eth, df_unfiltered_eth, dt = load_addressable_file(f)  # Load file
    
    # Random Gini unfiltered
    mean_eth, ci_eth_low, ci_eth_high = gini_random(df_unfiltered_eth["ETH"].to_numpy())
    mean_total, ci_total_low, ci_total_high = gini_random(df_unfiltered_total["Total"].to_numpy())

    # Filtered wallets data
    mask_removed_eth = df_unfiltered_eth["address"].str.strip_chars(" ").str.to_lowercase().is_in(all_labels_set)
    num_removed_eth = (mask_removed_eth).sum()
    eth_removed_pct = df_unfiltered_eth.filter(mask_removed_eth)["ETH"].sum() / df_unfiltered_eth["ETH"].sum()
    df_unfiltered_eth = df_unfiltered_eth.with_columns([pl.arange(0, df_unfiltered_eth.height).alias("Row")])
    filtered_wallets_eth = df_unfiltered_eth.filter(mask_removed_eth)
    if filtered_wallets_eth.height > 0: hist_eth,_ = np.histogram(filtered_wallets_eth["Row"].to_numpy(), bins=50, range=(0, 1_000_000))
    else: hist_eth = np.zeros(50)

    mask_removed_total = df_unfiltered_total["address"].str.strip_chars(" ").str.to_lowercase().is_in(all_labels_set)
    num_removed_total = (mask_removed_total).sum()
    total_removed_pct = df_unfiltered_total.filter(mask_removed_total)["Total"].sum() / df_unfiltered_total["Total"].sum()
    df_unfiltered_total = df_unfiltered_total.with_columns([pl.arange(0, df_unfiltered_total.height).alias("Row")])
    filtered_wallets_total = df_unfiltered_total.filter(mask_removed_total)
    if filtered_wallets_total.height > 0: hist_total,_ = np.histogram(filtered_wallets_total["Row"].to_numpy(), bins=50, range=(0, 1_000_000))
    else: hist_total = np.zeros(50)

    # Define rankings
    gini_rankings = {}
    for rank in rankings:
        gini_rankings[f"Ranking_{rank}_ETH"] = gini(df_unfiltered_eth[rank:].get_column("ETH").to_numpy())
        gini_rankings[f"Ranking_{rank}_Total"] = gini(df_unfiltered_total[rank:].get_column("Total").to_numpy())

    results_addressable.append({
        "Date": dt,

        # Gini Unfiltered & Filtered:
        "Gini_ETH_Unfiltered": gini(df_unfiltered_eth.get_column("ETH").to_numpy()),
        "Gini_Total_Unfiltered": gini(df_unfiltered_total.get_column("Total").to_numpy()),
        "Gini_ETH_Filtered": gini(df_filtered_eth.get_column("ETH").to_numpy()),
        "Gini_Total_Filtered": gini(df_filtered_total.get_column("Total").to_numpy()),
        
        # Random:
        "RandomMean_ETH": mean_eth,
        "RandomCIlow_ETH": ci_eth_low,
        "RandomCIhigh_ETH": ci_eth_high,
        "RandomMean_Total": mean_total,
        "RandomCIlow_Total": ci_total_low,
        "RandomCIhigh_Total": ci_total_high,

        # Ranking:
        **gini_rankings,

        # Filtered wallets data:
        "Filtered_Wallets_Number_eth": num_removed_eth,
        "Balance_Filtered_PCT_eth": eth_removed_pct,
        "Filtered_Row_Hist_eth": ",".join(map(str, hist_eth)),

        "Filtered_Wallets_Number_total": num_removed_total,
        "Balance_Filtered_PCT_total": total_removed_pct,
        "Filtered_Row_Hist_total": ",".join(map(str, hist_total))

    })

df_addressable_results = pl.DataFrame(results_addressable).sort("Date")


# -------------------- 02 - Compute Artemis Gini ---------------------
results_artemis = []

for f in artemis_files:
    df_filtered, df_unfiltered, dt = load_artemis_file(f) # Load file
    mean_eth, ci_eth_low, ci_eth_high = gini_random(df_unfiltered["ETH"].to_numpy()) # Random Gini unfiltered
    mean_eth_f, ci_eth_low_f, ci_eth_high_f = gini_random(df_filtered["ETH"].to_numpy()) # Random Gini filtered

    # Filtered wallets stats
    mask_removed = df_unfiltered["address"].str.strip_chars(" ").str.to_lowercase().is_in(all_labels_set)
    num_removed = (mask_removed).sum()
    eth_removed_pct = df_unfiltered.filter(mask_removed)["ETH"].sum() / df_unfiltered["ETH"].sum()

    # Compute ROWs and histogram 
    df_unfiltered = df_unfiltered.with_columns([pl.arange(0, df_unfiltered.height).alias("Row")])
    filtered_wallets = df_unfiltered.filter(mask_removed)
    
    # Histogram of filtered wallet ROWs
    if filtered_wallets.height > 0:
        hist, bin_edges = np.histogram(filtered_wallets["Row"].to_numpy(), bins=50, range=(0, 1_000_000))
    else:
        hist = np.zeros(50)

    # Define rankings
    gini_rankings = {}
    for rank in rankings:
        gini_rankings[f"Ranking_{rank}_ETH"] = gini(df_unfiltered[rank:].get_column("ETH").to_numpy())
        
    # Save results
    results_artemis.append({
    "Date": dt,

    # Gini Unfiltered & Filtered:
    "Gini_ETH_Unfiltered": gini(df_unfiltered.get_column("ETH").to_numpy()),
    "Gini_ETH_Filtered": gini(df_filtered.get_column("ETH").to_numpy()),
    
    # Random:
    "RandomMean_ETH": mean_eth,
    "RandomCIlow_ETH": ci_eth_low,
    "RandomCIhigh_ETH": ci_eth_high,
    "RandomMean_ETH_F": mean_eth_f,
    "RandomCIlow_ETH_F": ci_eth_low_f,
    "RandomCIhigh_ETH_F": ci_eth_high_f,

    # Ranking:
    **gini_rankings,
    
    # Filtered wallets data:
    "Filtered_Wallets_Number": num_removed,
    "ETH_Filtered_PCT": eth_removed_pct,
    "Filtered_Row_Hist": ",".join(map(str, hist))
    
})


df_artemis_results = pl.DataFrame(results_artemis).sort("Date")

# ------------------- 03 -  Export Results ---------------------------------
df_addressable_results.write_csv("Results/gini_addressable_results.csv")
df_artemis_results.write_csv("Results/gini_artemis_results.csv")

In [4]:
# -------------------------- Ranking Trend (Top 1M Wallets, Unfiltered) -----------------------
cut_points = [0, 50000, 100000, 150000, 200000, 250000]
file_names_ranking = [
    "2020-08-01T000000+0000.csv", "2021-08-01T000000+0000.csv", "2022-08-01T000000+0000.csv",
    "2023-08-01T000000+0000.csv", "2024-08-01T000000+0000.csv", "2025-08-01T000000+0000.csv"
]

def ranking_1M_UF(file_name):
    """ Returns the Gini coefficient based on the ranking trend for the top 1M wallets.
    The wallets are unfiltered. """
    file_path = next(f for f in artemis_files if os.path.basename(f) == file_name)
    df = pd.read_csv(file_path).nlargest(1000000, 'BALANCE_TOKEN')  # Include only the top 1M wallets
    ranking_gini_values_1M_UF = [gini(df.iloc[cut:]["BALANCE_TOKEN"].to_numpy()) for cut in cut_points] # Exclude the top x wallets
    parsed_file_name = os.path.splitext(os.path.basename(file_path))[0][:10]
    return parsed_file_name, ranking_gini_values_1M_UF

ranking_results_1M_UF_dict = {}

for file_name in file_names_ranking:
    key, gini_values = ranking_1M_UF(file_name)
    ranking_results_1M_UF_dict[key] = gini_values

df_results_ranking = pd.DataFrame.from_dict(ranking_results_1M_UF_dict, orient='index')
df_results_ranking.to_csv('Results/gini_ranking_results.csv')

In [5]:
# ----------------------------- Number of Wallets Holding More Than ETH Thresholds -----------------------
i_left = 0.1
i_right = 5000
row_counts_left = []
row_counts_right = []

for f in artemis_files:
    df = pd.read_csv(f, usecols=['BALANCE_TOKEN'])
    row_counts_left.append((df['BALANCE_TOKEN'] > i_left).sum())
    row_counts_right.append((df['BALANCE_TOKEN'] > i_right).sum())

df_counts = pd.DataFrame({
    "Date": df_artemis_results["Date"].to_list(),
    "Count_Left": row_counts_left,
    "Count_Right": row_counts_right})

df_counts.to_csv("Results/wallet_num_results.csv", index=False)

In [6]:
# ---------------- OVER 0.1 ETH - Number of wallets filtered & percentage of ETH filtered ------------------------

#  Preprocess once 
all_labels_set = {x.strip().lower() for x in all_labels_set}
num_wallets_filtered_list = []
eth_filtered_pct_list = []

# Loop 
for f in artemis_files:

    df = pd.read_csv(f,usecols=['ADDRESS', 'BALANCE_TOKEN'],
        dtype={'ADDRESS': 'string','BALANCE_TOKEN': 'float32'})

    # Normalize addresses
    addresses = df['ADDRESS'].str.strip().str.lower()

    # Fast boolean mask
    mask = addresses.isin(all_labels_set)
    balances_filtered = df.loc[mask, 'BALANCE_TOKEN']

    # Compute metrics
    num_wallets_filtered_list.append(len(balances_filtered))
    eth_filtered_pct_list.append(balances_filtered.sum() / df['BALANCE_TOKEN'].sum())

results_df = pd.DataFrame({
    'Date': df_artemis_results["Date"],
    'Num_Wallets_Filtered': num_wallets_filtered_list,
    'ETH_Filtered_Pct': eth_filtered_pct_list})

results_df.to_csv('Results/filtering_01_ETH_results.csv', index=False)

In [7]:
# --------------------- Mobility ---------------------------
num_groups = 5
labels = ["2021-2022", "2022-2023", "2023-2024", "2024-2025"]


def process_addressable_data_total_filtered(file_path, num_groups, valid_addresses=None):
    """Load and process addressable data"""
    df_filtered_total, _, _, _, _ = load_addressable_file(file_path)
    df_all = df_filtered_total.select(["address", "Total"]).to_pandas().sort_values("Total", ascending=False).reset_index(drop=True)
    df_all["Group"] = num_groups - pd.qcut(df_all["Total"].rank(method='first'), q=num_groups, labels=False)
    df_surv = None
    if valid_addresses is not None:
        df_surv = df_all[df_all["address"].isin(valid_addresses)].copy().reset_index(drop=True)
        df_surv["Group"] = num_groups - pd.qcut(df_surv["Total"].rank(method='first'), q=num_groups, labels=False)
    return df_all, df_surv

def process_artemis_data_filtered(file_path, num_groups, valid_addresses=None):
    """Load and process artemis data"""
    df_filtered, *_ = load_artemis_file(file_path)
    df_all = df_filtered.select(["address", "ETH"]).to_pandas().sort_values("ETH", ascending=False).reset_index(drop=True)
    df_all["Group"] = num_groups - pd.qcut(df_all["ETH"].rank(method='first'), q=num_groups, labels=False)
    df_surv = None
    if valid_addresses is not None:
        df_surv = df_all[df_all["address"].isin(valid_addresses)].copy().reset_index(drop=True)
        df_surv["Group"] = num_groups - pd.qcut(df_surv["ETH"].rank(method='first'), q=num_groups, labels=False)
    return df_all, df_surv

def get_common_addresses(files, process_func):
    """Find addresses present in all files."""
    common = None
    for f in files:
        df_all, _ = process_func(f, num_groups)
        addrs = set(df_all["address"])
        common = addrs if common is None else common.intersection(addrs)
    return list(common)

def get_transition_matrix(file_t0, file_t1, num_groups, as_percentages=False, process_func=None, valid_addresses=None):
    """Generate transition matrices for all and survivors."""
    df_all_t0, df_surv_t0 = process_func(file_t0, num_groups, valid_addresses)
    df_all_t1, df_surv_t1 = process_func(file_t1, num_groups, valid_addresses)
    
    def build(start, end):
        if start is None or end is None: return None
        merged = start.merge(end, on="address", how="left", suffixes=("_t0", "_t1"))
        merged["Group_t1"] = merged["Group_t1"].fillna("Exit")
        mat = merged.groupby(["Group_t0", "Group_t1"]).size().unstack(fill_value=0)
        cols = list(range(1, num_groups + 1)) + ["Exit"]
        mat = mat.reindex(index=list(range(1, num_groups + 1)), columns=cols, fill_value=0)
        return mat.div(mat.sum(axis=1), axis=0) * 100 if as_percentages else mat

    return build(df_all_t0, df_all_t1), build(df_surv_t0, df_surv_t1)

def extract_probabilities(files, num_groups, process_func):
    """Extract stay and exit probabilities per group."""
    df_same = pd.DataFrame(index=range(1, num_groups+1), columns=labels, dtype=float)
    df_exit = pd.DataFrame(index=range(1, num_groups+1), columns=labels, dtype=float)
    
    for i, col in enumerate(labels):
        mat, _ = get_transition_matrix(files[i], files[i+1], num_groups, as_percentages=True, process_func=process_func)
        for d in range(1, num_groups+1):
            df_same.loc[d, col] = mat.loc[d, d]
            df_exit.loc[d, col] = mat.loc[d, "Exit"]
    return df_same, df_exit

def extract_macro_statistics(files, num_groups, process_func):
    """Calculate mobility statistics and indices."""
    stats = {'labels': labels, 'shorrocks': [], 'shorrocks_survivors': [], 'exits_rate': [],
        'pct_up': [], 'pct_stable': [], 'pct_down': [], 'bartholomew': [], 'bartholomew_surv': []}
    survivors = get_common_addresses(files, process_func)
    row_i, col_i = np.arange(1, num_groups + 1)[:, None], np.arange(1, num_groups + 1)
    abs_diff = np.abs(row_i - col_i)

    for i in range(len(files) - 1):
        mat_all, mat_surv = get_transition_matrix(files[i], files[i+1], num_groups, as_percentages=False, process_func=process_func, valid_addresses=survivors)
        total = mat_all.sum().sum()
        stats['exits_rate'].append((mat_all["Exit"].sum() / total) * 100 if total else 0)
        
        # All users calculations
        stayers_all = mat_all.drop(columns=["Exit"])
        P_all = stayers_all.div(stayers_all.sum(axis=1), axis=0).fillna(0)
        stats['shorrocks'].append((num_groups - np.trace(P_all)) / (num_groups - 1))
        pi_all = stayers_all.sum(axis=1) / stayers_all.sum().sum()
        stats['bartholomew'].append((P_all.mul(pi_all, axis=0).values * abs_diff).sum())
        
        # Survivors calculations
        stayers_surv = mat_surv.drop(columns=["Exit"])
        P_surv = stayers_surv.div(stayers_surv.sum(axis=1), axis=0).fillna(0)
        stats['shorrocks_survivors'].append((num_groups - np.trace(P_surv)) / (num_groups - 1))
        pi_surv = stayers_surv.sum(axis=1) / stayers_surv.sum().sum()
        stats['bartholomew_surv'].append((P_surv.mul(pi_surv, axis=0).values * abs_diff).sum())
        
        # Population flows
        up, down, stable = 0, 0, 0
        for r in range(1, num_groups + 1):
            for c in range(1, num_groups + 1):
                count = mat_all.loc[r, c]
                shift = r - c
                if shift > 0: up += count
                elif shift < 0: down += count
                else: stable += count
        stats['pct_up'].append((up / total) * 100)
        stats['pct_down'].append((down / total) * 100)
        stats['pct_stable'].append((stable / total) * 100)

    return stats

# ----------------------- Execution  -----------------------------
# Addressable Data (Total Filtered) 
addressable_files = [
    "Data/Addressable_Data/1627776000.csv",
    "Data/Addressable_Data/1659312000.csv",
    "Data/Addressable_Data/1690848000.csv",
    "Data/Addressable_Data/1722470400.csv",
    "Data/Addressable_Data/1754006400.csv"]

addressable_df_same, addressable_df_exit = extract_probabilities(addressable_files, num_groups, process_func=process_addressable_data_total_filtered)
addressable_df_same.to_csv("Results/mobility_addressable_same.csv", index=False)
addressable_df_exit.to_csv("Results/mobility_addressable_exit.csv", index=False)
addressable_stats = extract_macro_statistics(addressable_files, num_groups, process_func=process_addressable_data_total_filtered)
pd.DataFrame(addressable_stats).to_csv("Results/mobility_addressable_stats.csv", index=False)

# Artemis Data (ETH Filtered)
artemis_files = [
    "Data/Artemis_Data/2021-08-01T000000+0000.csv",
    "Data/Artemis_Data/2022-08-01T000000+0000.csv",
    "Data/Artemis_Data/2023-08-01T000000+0000.csv",
    "Data/Artemis_Data/2024-08-01T000000+0000.csv",
    "Data/Artemis_Data/2025-08-01T000000+0000.csv"]

artemis_df_same, artemis_df_exit = extract_probabilities(artemis_files, num_groups, process_func=process_artemis_data_filtered)
artemis_df_same.to_csv("Results/mobility_artemis_same.csv", index=False)
artemis_df_exit.to_csv("Results/mobility_artemis_exit.csv", index=False)
artemis_stats = extract_macro_statistics(artemis_files, num_groups, process_func=process_artemis_data_filtered)
pd.DataFrame(artemis_stats).to_csv("Results/mobility_artemis_stats.csv", index=False)